# 03 - 게시자로 레코드 게시

이 Notebook에서는 AWS Agent Registry의 **게시자** 페르소나 워크플로를 보여 줍니다.
게시자는 MCP/A2A/CUSTOM/Agent Skills 레코드를 생성하고, 레코드를 나열하고 조회하며, 레코드 콘텐츠를 업데이트하고,
관리자 승인을 요청할 수 있습니다. 그러나 레코드를 **승인, 거부 또는 사용 중단 처리할 수는 없습니다.**

### 학습 내용

1. **나열 및 조회** — 게시자에게 표시되는 레지스트리와 레코드 탐색
2. **레코드 생성** — 메타데이터와 설명자가 포함된 MCP/A2A/CUSTOM 레코드 생성
3. **레코드 업데이트** — DRAFT 레코드의 설명자 수정
4. **승인 요청** — 레코드를 DRAFT → PENDING_APPROVAL로 전환

### 사전 요구 사항

- boto3 >= 1.42.87
- 관리자, 게시자, 소비자 페르소나용 IAM 역할을 생성하려면 [Notebook 01](01-create-user-personas-workflow.ipynb)을 실행하세요.
- 관리자로 레지스트리를 생성하려면 [Notebook 02](02-creating-registry-workflow.ipynb)를 실행하세요.


### 게시자 워크플로
![게시자 워크플로](images/publisher_flow_architecture.png)

### 게시자 API 참조

| # | API | 설명 |
|---|-----|-------------|
| 1 | [ListRegistries](https://docs.aws.amazon.com/boto3/latest/reference/services/bedrock-agentcore-control/client/list_registries.html) | 레코드를 게시할 수 있는 레지스트리 검색 |
| 2 | [GetRegistry](https://docs.aws.amazon.com/boto3/latest/reference/services/bedrock-agentcore-control/client/get_registry.html) | 레지스트리 세부 정보(이름, 상태, 승인 구성) 가져오기 |
| 3 | [CreateRegistryRecord](https://docs.aws.amazon.com/boto3/latest/reference/services/bedrock-agentcore-control/client/create_registry_record.html) | 새 MCP 또는 A2A 레코드 생성(CREATING → DRAFT) |
| 4 | [ListRegistryRecords](https://docs.aws.amazon.com/boto3/latest/reference/services/bedrock-agentcore-control/client/list_registry_records.html) | 레코드 나열 및 선택적으로 상태별 필터링 |
| 5 | [GetRegistryRecord](https://docs.aws.amazon.com/boto3/latest/reference/services/bedrock-agentcore-control/client/get_registry_record.html) | 설명자를 포함한 전체 레코드 세부 정보 가져오기 |
| 6 | [UpdateRegistryRecord](https://docs.aws.amazon.com/boto3/latest/reference/services/bedrock-agentcore-control/client/update_registry_record.html) | DRAFT 레코드 업데이트(`optionalValue` 래퍼를 사용하는 PATCH) |
| 7 | [SubmitRegistryRecordForApproval](https://docs.aws.amazon.com/boto3/latest/reference/services/bedrock-agentcore-control/client/submit_registry_record_for_approval.html) | 레코드를 DRAFT → PENDING_APPROVAL로 전환 |
| 8 | [DeleteRegistryRecord](https://docs.aws.amazon.com/boto3/latest/reference/services/bedrock-agentcore-control/client/delete_registry_record.html) | 레코드 삭제(게시자는 자신이 생성한 레코드를 삭제할 수 있음) |

### Notebook 진행 순서

02(레지스트리 생성) → **03(이 Notebook)** → 04(관리자 승인) → 05(시맨틱 검색)

#### 사용 사례: 엔터프라이즈 결제 처리
**게시자 페르소나:** AnyCompany의 결제 서비스 팀은 Payment Processing MCP Server, Loan Processing A2A Agent 및 기타 도구를 구축했습니다. 모든 레코드를 Agent Registry에 제출하여 관리자 승인을 받을 수 있으므로, 결제, 대출 및 기타 기능은 엔터프라이즈 전반의 다른 에이전트가 검색할 수 있게 되기 전에 검토를 거칩니다.

---
## 1. boto3 SDK 및 종속성 설치

핵심 종속성(`boto3` 및 `python-dotenv`)을 설치합니다.

In [ ]:
!pip install boto3 python-dotenv --force-reinstall

## 2. 게시자 역할을 수임하여 boto3 Session 초기화

`publisher_persona` IAM 역할을 수임하고 임시 자격 증명으로 boto3 Session을 생성합니다. 이후의 모든 API 호출은 이 Session을 사용합니다.

In [ ]:
import boto3
import json
import utils
import os
from botocore.exceptions import ClientError

AWS_REGION = os.environ.get("AWS_DEFAULT_REGION", "us-west-2")

# 현재 자격 증명에서 계정 ID 자동 감지
sts = boto3.client("sts", region_name=AWS_REGION)
ACCOUNT_ID = sts.get_caller_identity()["Account"]
CALLER_ARN = sts.get_caller_identity()["Arn"]

PUBLISHER_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/publisher_persona"

print(f"Account:  {PUBLISHER_ROLE_ARN}")

# 관리자 역할 수임
creds = utils.assume_role(
    role_arn=PUBLISHER_ROLE_ARN,
    session_name="publisher_session",
)

publisher_session = boto3.Session(
    aws_access_key_id=creds["AccessKeyId"],
    aws_secret_access_key=creds["SecretAccessKey"],
    aws_session_token=creds["SessionToken"],
    region_name=AWS_REGION,
)

## 3. Control Plane 클라이언트 초기화

Control plane(`bedrock-agentcore-control`)은 레지스트리와 레코드에 대한 CRUD 작업을 처리합니다.

In [ ]:
# Control plane 클라이언트(관리자 작업)
cp_client = publisher_session.client("bedrock-agentcore-control")

---
## 4. 게시자가 레지스트리 나열

레코드를 게시할 수 있는 레지스트리를 찾습니다.

In [ ]:
registries = cp_client.list_registries()
print(f"Publisher can see {len(registries.get('registries', []))} registries:\n")
for reg in registries.get("registries", []):
    utils.pp(reg)

## 5. 레지스트리 선택

사용할 기존 READY 레지스트리를 찾습니다. REGISTRY_ID를 설정하면 해당 값을 검증합니다. 설정하지 않으면 list_registries에서 첫 번째 READY 레지스트리를 선택합니다.

READY 레지스트리를 찾을 수 없다면 먼저 Notebook 02를 실행하여 레지스트리를 생성해야 합니다.

In [ ]:
REGISTRY_ID = ""  # 위 목록에서 직접 선택하려면 이 값을 입력

## REGISTRY_ID가 비어 있으면 list_registries에서 첫 번째 READY 레지스트리를 선택
registry_details = utils.get_or_select_registry(cp_client, REGISTRY_ID, AWS_REGION)
REGISTRY_ID = registry_details[0]
REGISTRY_ARN = registry_details[1]


## 6. MCP 레코드 생성

서버 및 도구 설명자가 포함된 MCP 레지스트리 레코드를 생성합니다. 레코드는 `CREATING` 상태로 시작하며 준비가 완료되면 `DRAFT`로 전환됩니다.

> 이 데이터는 데모용 샘플 데이터입니다.

### 6.1 MCP 설명자 스키마 정의

MCP 서버 메타데이터와 도구 입력 스키마를 정의합니다. 소비자가 검색하게 될 결제 처리 기능을 설명하는 정보입니다.

In [ ]:
mcp_server_schema = json.dumps(
    {
        "name": "io.novacorp/payment-processing-server",
        "description": "A payment processing MCP server for handling transactions, refunds, and payment status queries",
        "version": "1.0.0",
        "title": "Payment Processing Server",
        "packages": [
            {
                "registryType": "npm",
                "identifier": "@novacorp/payment-processing-mcp",
                "version": "1.0.0",
                "registryBaseUrl": "https://registry.npmjs.org",
                "runtimeHint": "npx",
                "transport": {"type": "stdio"},
            }
        ],
    }
)

mcp_tool_schema = json.dumps(
    {
        "tools": [
            {
                "name": "process_payment",
                "description": "Process a new payment transaction for a given amount and currency",
                "inputSchema": {
                    "type": "object",
                    "properties": {
                        "amount": {"type": "number", "description": "Payment amount"},
                        "currency": {
                            "type": "string",
                            "description": "ISO 4217 currency code (e.g. USD, EUR)",
                        },
                        "customer_id": {
                            "type": "string",
                            "description": "Unique customer identifier",
                        },
                        "payment_method": {
                            "type": "string",
                            "description": "Payment method type",
                            "enum": [
                                "credit_card",
                                "debit_card",
                                "bank_transfer",
                                "digital_wallet",
                            ],
                        },
                        "description": {
                            "type": "string",
                            "description": "Optional payment description",
                        },
                    },
                    "required": ["amount", "currency", "customer_id", "payment_method"],
                },
            },
            {
                "name": "get_payment_status",
                "description": "Retrieve the current status of a payment by transaction ID",
                "inputSchema": {
                    "type": "object",
                    "properties": {
                        "transaction_id": {
                            "type": "string",
                            "description": "Unique transaction identifier",
                        },
                    },
                    "required": ["transaction_id"],
                },
            },
            {
                "name": "process_refund",
                "description": "Initiate a full or partial refund for a completed transaction",
                "inputSchema": {
                    "type": "object",
                    "properties": {
                        "transaction_id": {
                            "type": "string",
                            "description": "Original transaction ID to refund",
                        },
                        "amount": {
                            "type": "number",
                            "description": "Refund amount (omit for full refund)",
                        },
                        "reason": {
                            "type": "string",
                            "description": "Reason for the refund",
                        },
                    },
                    "required": ["transaction_id", "reason"],
                },
            },
        ]
    }
)

### 6.2 MCP 레지스트리 레코드 생성

MCP 레코드를 레지스트리에 제출합니다.

> 참고: 이 셀을 여러 번 실행하면 고유 ID를 가진 중복 레코드가 생성됩니다.

In [ ]:
MCP_RECORD_ID = None

try:
    mcp_resp = cp_client.create_registry_record(
        registryId=REGISTRY_ID,
        name="mcp_payment_processing_server",
        description="MCP server for processing payments, refunds, and transaction queries",
        descriptorType="MCP",
        descriptors={
            "mcp": {
                "server": {
                    "schemaVersion": "2025-12-11",
                    "inlineContent": mcp_server_schema,
                },
                "tools": {
                    "inlineContent": mcp_tool_schema,
                },
            }
        },
        recordVersion="1.0",
    )
    MCP_RECORD_ID = mcp_resp["recordArn"].split("/")[-1]
    print(f"Created MCP record: {MCP_RECORD_ID}")
    record = utils.wait_for_record_ready(cp_client, REGISTRY_ID, MCP_RECORD_ID)
    print(f"Status: {record.get('status', 'UNKNOWN')}")

except ClientError as e:
    if e.response["Error"]["Code"] == "ConflictException":
        print("Record 'payment_processing_server' already exists — looking it up...")
        records = cp_client.list_registry_records(registryId=REGISTRY_ID)
        for rec in records.get("registryRecords", []):
            if rec["name"] == "payment_processing_server":
                MCP_RECORD_ID = rec["registryRecordId"]
                break
        print(f"  Using existing record: {MCP_RECORD_ID}")
    else:
        raise

print(f"\nMCP_RECORD_ID = {MCP_RECORD_ID}")

## 7. A2A 레코드 생성

Loan Processing Agent의 agent card 설명자가 포함된 A2A(Agent-to-Agent) 레지스트리 레코드를 생성합니다.

> 참고: 이 셀을 여러 번 실행하면 고유 ID를 가진 중복 레코드가 생성됩니다.

In [ ]:
from botocore.exceptions import ClientError

a2a_agent_card = json.dumps(
    {
        "protocolVersion": "0.3.0",
        "name": "Loan Processing Agent",
        "description": "Publishes loan processing capabilities for agent discovery, including loan applications, credit checks, and installment plan support.",
        "url": "https://example.com/agents/loan",
        "version": "1.0.0",
        "capabilities": {"streaming": True},
        "defaultInputModes": ["text"],
        "defaultOutputModes": ["text"],
        "preferredTransport": "JSONRPC",
        "skills": [
            {
                "id": "loan_application_processing",
                "name": "Loan Application Processing",
                "description": "Process loan applications and validate required information.",
                "tags": [],
            },
            {
                "id": "credit_check_review",
                "name": "Credit Check Review",
                "description": "Support credit check workflows and eligibility review.",
                "tags": [],
            },
            {
                "id": "installment_plan_management",
                "name": "Installment Plan Management",
                "description": "Manage installment plan setup and repayment schedule inquiries.",
                "tags": [],
            },
        ],
    }
)

try:
    a2a_resp = cp_client.create_registry_record(
        registryId=REGISTRY_ID,
        name="a2a_loan_agent",
        description="A2A agent for loan processing capabilities including loan applications, credit checks, and installment plan support.",
        descriptorType="A2A",
        descriptors={
            "a2a": {
                "agentCard": {
                    "schemaVersion": "0.3",
                    "inlineContent": a2a_agent_card,
                }
            }
        },
        recordVersion="1.0",
    )
    A2A_RECORD_ID = a2a_resp["recordArn"].split("/")[-1]  # registryRecordArn이 아니라 recordArn
    print(f"Created A2A record: {A2A_RECORD_ID}")
    utils.wait_for_record_ready(cp_client, REGISTRY_ID, A2A_RECORD_ID)

except ClientError as e:
    if e.response["Error"]["Code"] == "ConflictException":
        print("Record 'payment_agent' already exists — looking it up...")
        records = cp_client.list_registry_records(registryId=REGISTRY_ID)
        for rec in records.get("registryRecords", []):
            if rec["name"] == "payment_agent":
                A2A_RECORD_ID = rec["recordId"]  # registryRecordId가 아니라 recordId
                break
        print(f"  Using existing record: {A2A_RECORD_ID}")
    else:
        raise

print(f"\nA2A_RECORD_ID = {A2A_RECORD_ID}")

## 8. 추가 레코드 생성

환불 분석, 신용 평가, 사기 탐지, 청구 분쟁, 결제 조정을 다루는 레코드 5개를 추가로 생성합니다. 이러한 레코드는 Notebook 05의 검색 데모를 위해 레지스트리를 보강합니다.

> 참고: 이 셀을 여러 번 실행하면 고유 ID를 가진 중복 레코드가 생성됩니다.

In [ ]:
ADDITIONAL_RECORDS = [
    {
        "name": "mcp_refund_analytics_server",
        "description": "MCP server for refund trend analysis, chargeback reporting, and refund policy compliance checks.",
        "descriptorType": "MCP",
        "descriptors": {
            "mcp": {
                "server": {
                    "schemaVersion": "2025-12-11",
                    "inlineContent": json.dumps(
                        {
                            "name": "io.novacorp/refund-analytics-server",
                            "description": "Analyzes refund patterns, chargeback rates, and policy compliance",
                            "version": "1.0.0",
                        }
                    ),
                },
                "tools": {
                    "inlineContent": json.dumps(
                        {
                            "tools": [
                                {
                                    "name": "get_refund_trends",
                                    "description": "Analyze refund trends over a date range",
                                    "inputSchema": {
                                        "type": "object",
                                        "properties": {
                                            "start_date": {"type": "string"},
                                            "end_date": {"type": "string"},
                                        },
                                        "required": ["start_date"],
                                    },
                                },
                                {
                                    "name": "check_chargeback_rate",
                                    "description": "Get chargeback rate for a merchant",
                                    "inputSchema": {
                                        "type": "object",
                                        "properties": {"merchant_id": {"type": "string"}},
                                        "required": ["merchant_id"],
                                    },
                                },
                            ]
                        }
                    ),
                },
            }
        },
        "recordVersion": "1.0",
    },
    {
        "name": "a2a_credit_score_agent",
        "description": "A2A agent for real-time credit score retrieval, credit history analysis, and risk assessment for lending decisions.",
        "descriptorType": "A2A",
        "descriptors": {
            "a2a": {
                "agentCard": {
                    "schemaVersion": "0.3",
                    "inlineContent": json.dumps(
                        {
                            "protocolVersion": "0.3.0",
                            "name": "Credit Score Agent",
                            "description": "Retrieves credit scores, analyzes credit history, and provides risk assessments for lending workflows.",
                            "url": "https://example.com/agents/credit-score",
                            "version": "1.0.0",
                            "capabilities": {"streaming": False},
                            "defaultInputModes": ["text"],
                            "defaultOutputModes": ["text"],
                            "skills": [
                                {
                                    "id": "get_credit_score",
                                    "name": "Get Credit Score",
                                    "description": "Retrieve current credit score for a customer.",
                                    "tags": [],
                                },
                                {
                                    "id": "credit_risk_assessment",
                                    "name": "Credit Risk Assessment",
                                    "description": "Evaluate lending risk based on credit history.",
                                    "tags": [],
                                },
                            ],
                        }
                    ),
                }
            }
        },
        "recordVersion": "1.0",
    },
    {
        "name": "mcp_fraud_detection_server",
        "description": "MCP server for real-time transaction fraud detection, suspicious activity flagging, and fraud case management.",
        "descriptorType": "MCP",
        "descriptors": {
            "mcp": {
                "server": {
                    "schemaVersion": "2025-12-11",
                    "inlineContent": json.dumps(
                        {
                            "name": "io.novacorp/fraud-detection-server",
                            "description": "Detects fraudulent transactions and manages fraud cases",
                            "version": "1.0.0",
                        }
                    ),
                },
                "tools": {
                    "inlineContent": json.dumps(
                        {
                            "tools": [
                                {
                                    "name": "scan_transaction",
                                    "description": "Scan a transaction for fraud indicators",
                                    "inputSchema": {
                                        "type": "object",
                                        "properties": {"transaction_id": {"type": "string"}},
                                        "required": ["transaction_id"],
                                    },
                                },
                                {
                                    "name": "flag_suspicious_activity",
                                    "description": "Flag an account for suspicious activity review",
                                    "inputSchema": {
                                        "type": "object",
                                        "properties": {
                                            "account_id": {"type": "string"},
                                            "reason": {"type": "string"},
                                        },
                                        "required": ["account_id", "reason"],
                                    },
                                },
                            ]
                        }
                    ),
                },
            }
        },
        "recordVersion": "1.0",
    },
    {
        "name": "a2a_billing_dispute_agent",
        "description": "A2A agent for handling billing disputes, charge contestations, and resolution tracking across customer accounts.",
        "descriptorType": "A2A",
        "descriptors": {
            "a2a": {
                "agentCard": {
                    "schemaVersion": "0.3",
                    "inlineContent": json.dumps(
                        {
                            "protocolVersion": "0.3.0",
                            "name": "Billing Dispute Agent",
                            "description": "Manages billing disputes, charge contestations, and tracks resolution status.",
                            "url": "https://example.com/agents/billing-dispute",
                            "version": "1.0.0",
                            "capabilities": {"streaming": True},
                            "defaultInputModes": ["text"],
                            "defaultOutputModes": ["text"],
                            "skills": [
                                {
                                    "id": "open_dispute",
                                    "name": "Open Dispute",
                                    "description": "Open a new billing dispute for a customer charge.",
                                    "tags": [],
                                },
                                {
                                    "id": "track_resolution",
                                    "name": "Track Resolution",
                                    "description": "Check the status of an ongoing billing dispute.",
                                    "tags": [],
                                },
                            ],
                        }
                    ),
                }
            }
        },
        "recordVersion": "1.0",
    },
    {
        "name": "mcp_payment_reconciliation_server",
        "description": "MCP server for reconciling payment records across systems, identifying discrepancies, and generating settlement reports.",
        "descriptorType": "MCP",
        "descriptors": {
            "mcp": {
                "server": {
                    "schemaVersion": "2025-12-11",
                    "inlineContent": json.dumps(
                        {
                            "name": "io.novacorp/payment-reconciliation-server",
                            "description": "Reconciles payments across systems and generates settlement reports",
                            "version": "1.0.0",
                        }
                    ),
                },
                "tools": {
                    "inlineContent": json.dumps(
                        {
                            "tools": [
                                {
                                    "name": "reconcile_payments",
                                    "description": "Reconcile payment records between two systems for a date range",
                                    "inputSchema": {
                                        "type": "object",
                                        "properties": {
                                            "source_system": {"type": "string"},
                                            "target_system": {"type": "string"},
                                            "date": {"type": "string"},
                                        },
                                        "required": [
                                            "source_system",
                                            "target_system",
                                            "date",
                                        ],
                                    },
                                },
                                {
                                    "name": "generate_settlement_report",
                                    "description": "Generate a settlement report for a merchant",
                                    "inputSchema": {
                                        "type": "object",
                                        "properties": {
                                            "merchant_id": {"type": "string"},
                                            "period": {"type": "string"},
                                        },
                                        "required": ["merchant_id", "period"],
                                    },
                                },
                            ]
                        }
                    ),
                },
            }
        },
        "recordVersion": "1.0",
    },
    {
        "name": "custom_payment_gateway_config",
        "description": "Custom descriptor for AnyCompany's payment gateway configuration, including supported providers, retry policies, and regional routing rules.",
        "descriptorType": "CUSTOM",
        "descriptors": {
            "custom": {
                "inlineContent": json.dumps(
                    {
                        "gatewayName": "NovaCorp Payment Gateway",
                        "version": "2.1.0",
                        "supportedProviders": ["stripe", "adyen", "square"],
                        "endpoint": "https://gateway.novacorp.example.com/v2",
                        "retryPolicy": {"maxRetries": 3, "backoffMs": 500},
                        "regionalRouting": {
                            "us": "us-east-1",
                            "eu": "eu-west-1",
                            "apac": "ap-southeast-1",
                        },
                    }
                ),
            }
        },
        "recordVersion": "1.0",
    },
]

additional_record_ids = []
for rec in ADDITIONAL_RECORDS:
    try:
        resp = cp_client.create_registry_record(registryId=REGISTRY_ID, **rec)
        rid = resp["recordArn"].split("/")[-1]
        additional_record_ids.append(rid)
        print(f"  ✅ Created [{rec['descriptorType']}] {rec['name']} — {rid}")
        utils.wait_for_record_ready(cp_client, REGISTRY_ID, rid)
    except ClientError as e:
        if e.response["Error"]["Code"] == "ConflictException":
            print(f"  ⚠️  {rec['name']} already exists — skipping.")
        else:
            print(f"  ❌ Failed: {rec['name']} — {e}")

print(f"\nCreated {len(additional_record_ids)} additional records.")


## 9. 레지스트리의 레코드 나열

레지스트리의 모든 레코드를 나열합니다. 결과 범위를 좁히려면 상태 또는 설명자 유형으로 필터링합니다.

In [ ]:
# 모든 레코드 나열

records_resp = cp_client.list_registry_records(registryId=REGISTRY_ID)

print(f"Total records in registry: {len(records_resp.get('registryRecords', []))}\n")
for rec in records_resp.get("registryRecords", []):
    print(f"  {rec['name']} | {rec['recordId']} |  {rec['descriptorType']} | {rec['status']}")


## 10. 레코드 세부 정보 가져오기

설명자를 포함하여 특정 레코드의 전체 세부 정보를 가져옵니다.

In [ ]:
# 단일 MCP 레코드 세부 정보 가져오기
mcp_detail = cp_client.get_registry_record(
    registryId=REGISTRY_ID,
    recordId=MCP_RECORD_ID,
)

print("MCP Record Details:")
utils.pp(mcp_detail)

agent card 설명자를 포함한 A2A 레코드의 세부 정보를 가져옵니다.

In [ ]:
# 단일 A2A 레코드 세부 정보 가져오기
a2a_detail = cp_client.get_registry_record(
    registryId=REGISTRY_ID,
    recordId=A2A_RECORD_ID,
)

print("A2A Record Details:")
utils.pp(a2a_detail)


## 11. 레코드 업데이트

`DRAFT` 상태의 레코드는 업데이트할 수 있습니다. `UpdateRegistryRecord` API는 `optionalValue` 래퍼를 사용하는 PATCH 의미 체계를 따릅니다.

여기서는 MCP 도구 스키마에 `list_transactions` 도구를 추가합니다.

In [ ]:
updated_tool_schema = json.dumps(
    {
        "tools": [
            {
                "name": "list_transactions",
                "description": "List payment transactions for a customer with optional date filtering",
                "inputSchema": {
                    "type": "object",
                    "properties": {
                        "customer_id": {
                            "type": "string",
                            "description": "Unique customer identifier",
                        },
                        "start_date": {
                            "type": "string",
                            "description": "Start date filter (ISO 8601)",
                        },
                        "end_date": {
                            "type": "string",
                            "description": "End date filter (ISO 8601)",
                        },
                        "status": {
                            "type": "string",
                            "description": "Filter by transaction status",
                            "enum": ["pending", "completed", "failed", "refunded"],
                        },
                    },
                    "required": ["customer_id"],
                },
            }
        ]
    }
)

update_resp = cp_client.update_registry_record(
    registryId=REGISTRY_ID,
    recordId=MCP_RECORD_ID,
    descriptors={
        "optionalValue": {
            "mcp": {
                "optionalValue": {
                    "tools": {
                        "optionalValue": {
                            "inlineContent": updated_tool_schema,
                        }
                    }
                }
            }
        }
    },
)

print("Update submitted — waiting for record to settle...")
updated_record = utils.wait_for_record_ready(cp_client, REGISTRY_ID, MCP_RECORD_ID)
print("\nUpdated record:")
utils.pp(updated_record)


## 12. 게시자가 "DRAFT" 상태인 모든 레코드의 관리자 승인 요청

`SubmitRegistryRecordForApproval`을 사용하여 레코드를 `DRAFT` → `PENDING_APPROVAL`로 전환합니다.


In [ ]:
# 모든 DRAFT 레코드의 승인 요청(MCP + A2A)
draft_resp = cp_client.list_registry_records(
    registryId=REGISTRY_ID,
    status="DRAFT",
)
draft_records = draft_resp.get("registryRecords", [])
print(f"Found {len(draft_records)} DRAFT records\n")

for rec in draft_records:
    record_id = rec["recordId"]
    try:
        submit_resp = cp_client.submit_registry_record_for_approval(
            registryId=REGISTRY_ID,
            recordId=record_id,
        )
        print(f"Submitted: {rec['name']} ({rec['descriptorType']}) — {record_id}")
        utils.wait_for_record_ready(cp_client, REGISTRY_ID, record_id)
    except ClientError as e:
        error_code = e.response["Error"]["Code"]
        error_msg = e.response["Error"].get("Message", "")
        print(f"Failed: {rec['name']} ({record_id}) — {error_code}: {error_msg}")
    print()

### 대기 중인 레코드 확인

제출한 모든 레코드가 이제 `PENDING_APPROVAL` 상태인지 확인합니다.

In [ ]:
# PENDING_APPROVAL 상태의 모든 레코드 확인

pending_resp = cp_client.list_registry_records(
    registryId=REGISTRY_ID,
    status="PENDING_APPROVAL",
)
pending_records = pending_resp.get("registryRecords", [])
print(f"Records in PENDING_APPROVAL: {len(pending_records)}\n")

for rec in pending_records:
    print(f"  {rec['name']} ({rec['descriptorType']}) — {rec['recordId']}: {rec['status']}")


## 13. 정리

이 Notebook에서 생성한 데모 리소스를 제거합니다.

⚠️ 삭제를 진행하려면 아래 코드의 주석을 해제하세요.

In [ ]:
# # 레코드 삭제(게시자는 자신이 생성한 레코드를 삭제할 수 있음)
# for rid, label in [(MCP_RECORD_ID, "MCP"), (A2A_RECORD_ID, "A2A")]:
#     try:
#         cp_client.delete_registry_record(
#             registryId=REGISTRY_ID, recordId=rid
#         )
#         print(f"Deleted {label} record: {rid}")
#     except Exception as e:
#         print(f"Record cleanup ({label}): {e}")

# print("\nCleanup complete!")

### ID로 특정 레코드 삭제

이 셀을 사용하여 레코드 ID로 특정 레코드를 삭제합니다. 제거할 ID로 `IDS_TO_DELETE`를 업데이트하세요.

In [ ]:
# IDS_TO_DELETE = ["EtW5lsG3TFBk"]  # 삭제할 레코드 ID로 변경

# # 레지스트리에서 현재 레코드 가져오기
# all_records = cp_client.list_registry_records(registryId=REGISTRY_ID).get("registryRecords", [])

# targets = [r for r in all_records if r.get("recordId") in IDS_TO_DELETE]
# print(f"{len(targets)} record(s) matched\n")
# for r in targets:
#     print(f"  {r.get('name')} — {r.get('recordId')}")

# # 일치하는 레코드 삭제
# for r in targets:
#     try:
#         cp_client.delete_registry_record(registryId=REGISTRY_ID, recordId=r["recordId"])
#         print(f"  Deleted: {r['name']} ({r['recordId']})")
#     except Exception as e:
#         print(f"  FAILED:  {r['name']} ({r['recordId']}) — {e}")

## 사전 요구 Notebook
- **Notebook 01** — [사용자 페르소나 생성](01-create-user-personas-workflow.ipynb): 관리자, 게시자, 소비자 사용자 페르소나 설정
- **Notebook 02** — [레지스트리 생성](02-creating-registry-workflow.ipynb): 관리자가 레지스트리 생성

## 다음 단계
- **Notebook 04** — [관리자 승인](04-admin-approval-workflow.ipynb): 관리자 승인 워크플로
- **Notebook 05** — [시맨틱 검색](05-search-registry-workflow.ipynb): 소비자로서 NLQ를 사용해 승인된 레코드 검색